**Importation des modules nécessaires**

In [ ]:
# Chargement des modules neccessaires
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt # 
import seaborn as sns 
import plotly.express as px
import scipy.stats as stat
from tabulate import tabulate
from scipy import stats 

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix

import shap
import warnings
warnings.filterwarnings('ignore') 
pd.options.mode.chained_assignment = None

**Gestion du temps**

In [ ]:
# Définition des tâches et durées
tache = {
    "Tâches": [
        "Etude du contexte",
        "Recherche & Traitement des données",
        "Analyse exploratoire des Données",
        "Modélisation ML",
        "Tests & validation",
        "Analyse des résultats", 
        "Rédaction & Publication"
    ],
    "Début": ["2025-03-17", "2025-03-25", "2025-04-02","2025-04-11", "2025-04-14", "2025-04-17", "2025-04-22"],
    "Fin": ["2025-03-23", "2025-03-31", "2025-04-10", "2025-04-13","2025-04-17", "2025-04-21", "2025-05-01"]
}

# Tracer le diagramme de Gantt
df = pd.DataFrame(tache)
title = "Diagramme de GANTT"
xaxis_title = "Dates d'exécution (semaines)"
yaxis_title = "Tâches à réaliser"

def gantt_diagram (df, title=None, xaxis_title=None, yaxis_title=None):
        
        """

        Il s'agit d'une fonction qui permet de générer un Diagramme de GANTT 
        montrant l'agencement du projet.

        """

        fig = px.timeline(df, x_start="Début", x_end="Fin", y="Tâches", color="Tâches",
            title=title, labels={"Task": yaxis_title})
        fig.update_yaxes(autorange="reversed")
        fig.update_layout(xaxis_title=xaxis_title)
        fig.show() 
    
    # Création du Gantt chart
gantt_diagram(df, title=title, xaxis_title=xaxis_title, yaxis_title=yaxis_title)

**0. Chargement des données**

In [ ]:
# Chatagement du dataset
data = pd.read_csv("fraud_data.csv", delimiter = ',')
df_5_row = data.head()
df_5_row

In [ ]:
# Information sur le dataset
print(f"Le nombre de clients : {data.shape[0]} \n")
print(f"Le nombre de caractéristiques : {data.shape[1]} \n")

**1. Analyse Univarée**

In [ ]:
class AnalyseUnivariee:
    def __init__(self, df):
        self.df = df
        self.var_quanti = df.select_dtypes(include=[float]).columns.tolist()
        self.var_quali = df.select_dtypes(include=[int, object, "category"]).columns.tolist()
        # On enlève les colonnes non pertinentes si elles existent
        for col in ["nameOrig", "step", "nameDest"]:
            if col in self.var_quali:
                self.var_quali.remove(col)

    def analyser_quanti(self):
        print(f"\n📊 Statistique descriptive des variables quantitatives :\n{self.df[self.var_quanti].describe()}\n")

        print("📦 Analyse de la dispersion (boxplots) :")
        for col in self.var_quanti:
            plt.figure(figsize=(10, 5))
            sns.boxplot(x=self.df[col])
            plt.title(f"Boxplot de {col}")
            plt.xlabel(col)
            plt.show()

        print("🔍 Analyse de la normalité (histogrammes + Shapiro) :")
        for col in self.var_quanti:
            sw, pvalue = stat.shapiro(self.df[col].dropna())  # on évite les NaN
            plt.figure(figsize=(10, 5))
            sns.histplot(self.df[col], kde=True)
            if pvalue < 0.05:
                plt.title(f"{col} n'est pas normalement distribuée (p = {np.round(pvalue, 2)})")
            else:
                plt.title(f"{col} est normalement distribuée (p = {np.round(pvalue, 2)})")
            plt.show()

    def analyser_quali(self):
        print(f"\n📋 Statistique descriptive des variables qualitatives :\n{self.df[self.var_quali].describe(include='all')}\n")

        for col in self.var_quali:
            counts = self.df[col].value_counts()
            plt.figure(figsize=(13, 8))
            plt.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=90, 
                    colors=['skyblue', 'salmon', 'lightgreen'])
            plt.title(f"Répartition de {col}")
            plt.show()

    def lancer_analyse(self):
        print("\n🔎 DÉBUT DE L'ANALYSE UNIVARIÉE\n")
        if self.var_quanti:
            self.analyser_quanti()
        if self.var_quali:
            self.analyser_quali()
        print("\n✅ FIN DE L'ANALYSE UNIVARIÉE\n")


In [ ]:
# Resultats des analyses
if __name__ == "__main__":
    print("\n🔎 DÉBUT DE L'ANALYSE UNIVARIÉE\n")
    # Instanciation de la classe AnalyseUnivariee
    # et lancement de l'analyse univariée
    analyse_1 = AnalyseUnivariee(data)

    # Analyse des variables quantitative
    analyse_1.analyser_quanti()

    # Analyses des variables qualitatives
    analyse_1.analyser_quali()

**2. Analyse Bivariée**

In [ ]:
class BivariateAnalysis:

    def __init__(self, df):
        self.df = df
        self.variables = df.columns.tolist()
        self.quant_vars = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
        self.qual_vars = ['type', 'isFraud']

    def BivariateQuant(self, variables):
        if not isinstance(variables, list) or len(variables) < 2:
            raise ValueError("'variables' doit être une liste d'au moins 2 variables")

        valid_vars = [var for var in variables if var in self.df.columns and pd.api.types.is_numeric_dtype(self.df[var])]

        if len(valid_vars) < 2:
            raise ValueError("Moins de 2 variables numériques valides trouvées dans le DataFrame")

        df_clean = self.df[valid_vars].dropna()

        for i in range(len(valid_vars)):
            for j in range(i + 1, len(valid_vars)):
                x_var, y_var = valid_vars[i], valid_vars[j]

                plt.figure(figsize=(8, 6))
                sns.regplot(data=df_clean, x=x_var, y=y_var,
                            scatter_kws={'alpha': 0.5},
                            line_kws={'color': 'red'})
                plt.title(f"Relation entre {x_var} et {y_var}", pad=20)
                plt.tight_layout()
                plt.show()

                corr, pvalue = stats.spearmanr(df_clean[x_var], df_clean[y_var])
                significance = "significative" if pvalue < 0.05 else "non significative"
                print(f"[{x_var} vs {y_var}] Corrélation {significance} (rho = {corr:.2f}, p = {pvalue:.4f})")
                print("-" * 50)

        corr_matrix = df_clean.corr(method='spearman')
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

        plt.figure(figsize=(10, 8))
        sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", square=True)
        plt.title("Matrice de corrélation (Spearman)", pad=20)
        plt.tight_layout()
        plt.show()

    def BivariateQual(self):
        for i in self.qual_vars:
            for j in self.qual_vars:
                if i != j:
                    contingency_table = pd.crosstab(self.df[i], self.df[j])
                    print(f"\n## Test du Chi2 entre {i} et {j} ##")
                    chi2, pvalue, dof, expected = stats.chi2_contingency(contingency_table)
                    if pvalue < 0.05:
                        print(f"➡ Association entre {i} et {j}, p = {pvalue:.4f}")
                    else:
                        print(f"✖ Aucune association entre {i} et {j}, p = {pvalue:.4f}")

    def BivariateQuantQual(self):
        quant_vars = [var for var in self.quant_vars if var in self.df.columns and pd.api.types.is_numeric_dtype(self.df[var])]
        qual_vars = [var for var in self.qual_vars if var in self.df.columns and not pd.api.types.is_numeric_dtype(self.df[var])]

        if not quant_vars or not qual_vars:
            raise ValueError("Au moins une variable quantitative et une qualitative sont nécessaires")

        df_clean = self.df[quant_vars + qual_vars].dropna()

        for quant_var in quant_vars:
            for qual_var in qual_vars:
                try:
                    groups = [group[quant_var].values for name, group in df_clean.groupby(qual_var)]
                    h_stat, pvalue = stats.kruskal(*groups)
                    significance = "significative" if pvalue < 0.05 else "non significative"

                    print(f"\nTest Kruskal-Wallis entre {quant_var} et {qual_var} :")
                    print(f"H = {h_stat:.2f}, p = {pvalue:.4f} → Association {significance}")

                    plt.figure(figsize=(10, 6))
                    sns.boxplot(x=qual_var, y=quant_var, data=df_clean)
                    sns.stripplot(x=qual_var, y=quant_var, data=df_clean,
                                  color='black', alpha=0.3, jitter=True)
                    plt.title(f"{quant_var} par {qual_var} (p = {pvalue:.4f})", pad=20)
                    plt.xticks(rotation=45)
                    plt.tight_layout()
                    plt.show()

                except ValueError as e:
                    print(f"\nErreur avec {quant_var} et {qual_var} : {e}")

    def lancer_analyse(self):
        print("\n--- Analyse bivariée quantitative ---")
        self.BivariateQuant(self.quant_vars)

        print("\n--- Analyse bivariée qualitative ---")
        self.BivariateQual()

        print("\n--- Analyse croisée quanti vs quali ---")
        self.BivariateQuantQual()

        print("\n✅ FIN DE L'ANALYSE  BIVARIÉE")


In [ ]:
if __name__ == "__main__":
    # Exemple de test rapide

    analyse = BivariateAnalysis(data[['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest','type', 'isFraud']])
    analyse.BivariateQuantQual() 

**4.1. Analyse Multivariée avant le Feature Engineering**

*   Encodage
*   Normalisation
*   Définition du target et des features
*   Instanciation de PCA

In [ ]:
# Suppression des variables inutiles
data_1 = data.copy()
data.drop(columns=["isFlaggedFraud","nameOrig","nameDest"], inplace=True) 

# Conversiond des variables en catégories
var_categorielle = ["isFraud","type"]
for var in var_categorielle:
    data[var] = data[var].astype("category")

# Encoding des variables catégorielles
df_demmies_0 = pd.get_dummies(data[["type"]], prefix=["type"], prefix_sep="_", dtype=int)
data_0 = pd.concat([data,df_demmies_0], axis=1)
data_0.drop(columns=["type"], inplace = True)

# Normalisation des varaibles
data_0[[
    'amount','oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest']] = (data_0[['amount','oldbalanceOrg', 'newbalanceOrig','oldbalanceDest', 'newbalanceDest']] - data_0[['amount','oldbalanceOrg', 'newbalanceOrig','oldbalanceDest', 'newbalanceDest']].mean())/data_0[['amount','oldbalanceOrg', 'newbalanceOrig','oldbalanceDest', 'newbalanceDest']].std()
data_vf = data_0.copy()

In [ ]:
class MultivariateAnalyser:
    def __init__(self, df):
        self.data = df
    
    def pca(self):

        # Définition des features et de la variables cibles
        features = self.data.drop(columns = "isFraud")
        target = self.data["isFraud"]
        pca = PCA()
        features_pca = pca.fit_transform(features)

      
        # Determination des composants principales
        composant_principale = pd.DataFrame(
            {
                "Dimension" : ["PCA_" + str(x) for x in range(features.shape[1])],
                "Valeur propre" : np.round(pca.explained_variance_,3),
                "Taux_variance" : np.round(pca.explained_variance_ratio_,3),
                "Taux_variance_cum" : np.round(np.cumsum(pca.explained_variance_ratio_),3)
            }
        )

        # Impact de variables initiales sur les composantes
        vecteur_propre = pd.DataFrame(
            pca.components_.T,
            columns=["PCA_" + str(x+1) for x in range(features.shape[1])],
            index = features.columns.to_list()
        )

        # Tri des variables par leur contribution absolue maximale à une composante␣principale
        vecteur_propre_sorted = vecteur_propre.abs().max(axis=1).sort_values(ascending=False).index
        vecteur_propre = vecteur_propre.loc[vecteur_propre_sorted]

        # Affichage du tableau croisé des 5 premières variables les plus contributives
        vecteur_propre_5 = vecteur_propre.sort_values(by="PCA_1", ascending=False)

        # Representation graphique de l'ACP

        # Créer le cercle de corrélation
        coeff = np.transpose(pca.components_[0:2, :])
        n = coeff.shape[0]
        xs = np.array([1, 0])
        ys = np.array([0, 1])
        feature_names = features.columns.to_list()
        # Créer la figure
        plt.figure(figsize=(10, 10))

        # Placer les vecteurs des variables
        for i in range(n):
            plt.arrow(0, 0, coeff[i, 0], coeff[i, 1], color='k', alpha=0.9, head_width=0.02)
            plt.text(coeff[i, 0] * 1.15, coeff[i, 1] * 1.15, feature_names[i], color='k', ha='center', va='center')

        # Placer le cercle unitaire
        circle = plt.Circle((0, 0), 1, color='gray', fill=False, linestyle='--')
        plt.gca().add_artist(circle)

        # Ajuster les limites et les axes
        plt.xlim(-1, 1)
        plt.ylim(-1, 1)
        plt.axhline(0, color='gray', linewidth=1)
        plt.axvline(0, color='gray', linewidth=1)
        plt.xlabel('PC1')
        plt.ylabel('PC2')
        plt.title('Cercle de corrélation ACP des fraudes')

        # Afficher la figure
        plt.show()
            
        return composant_principale, vecteur_propre_5
    
    def lancer_analyse(self):

        """
        Cela permet de lancer l'analyse
        """
        print("\n --Analyse Multivariée -- \n")
        cp, vp5 = self.pca()
        print("-- Les Composants Principaux --")
        print(f"\n {cp} \n")

        print("-- Influence de chaque variable dans les composants --")
        print(f"{vp5} \n")
        print("\n✅ FIN DE L'ANALYSE BIVARIÉE")

In [ ]:
an_before = MultivariateAnalyser(data_vf)
an_before.lancer_analyse()

**3. Feature Engineering**

- L'objetcif principal ici c'est de créer des variables dérivées permettant de capter au moins les anomalies

In [ ]:
# Variables dérivées
## | Vérifie si l'argent est vraiment retiré du compte client. Si ≠ 0 → anomalie possible. |
data_1["DiffSoldeOrig"] = data_1["oldbalanceOrg"] - data_1["amount"] - data_1["newbalanceOrig"] 
data_1["DiffSoldeOrig"] = data_1["DiffSoldeOrig"].apply(lambda x: 1 if x != 0 else 0) # | Si ≠ 0 → anomalie possible. |

## | Pareil mais côté bénéficiaire |
data_1["DiffSoldeDest"] = data_1["oldbalanceDest"] + data_1["amount"] - data["newbalanceDest"] 
data_1["DiffSoldeDest"] = data_1["DiffSoldeDest"].apply(lambda x : 1 if x != 0 else 0) # | Si ≠ 0 → anomalie possible. |

## | Montant transféré par rapport au solde initial du client. Un % élevé est suspect. |
data_1["PourcentageMontantOrig"] = data_1["amount"] / (data_1["oldbalanceOrg"] + 1) 

## | 1 si nameOrig == nameDest sinon 0 | Transaction entre le même compte : peut être une fraude déguisée. |
data_1["isSameAccount"] = data_1["nameOrig"] == data_1["nameDest"] 
data_1["isSameAccount"] = data_1["isSameAccount"].apply(lambda x : 1 if x == True else 0) # | 1 si nameOrig == nameDest sinon 0 | Transaction entre le même compte : peut être une fraude déguisée. |

## | activitéAnormale | 1 si (DiffSoldeOrig ≠ 0 OU DiffSoldeDest ≠ 0) sinon 0 | Comportement suspect basé sur les incohérences des soldes. |
data_1["activiteAnormale"] = data_1["DiffSoldeOrig"] + data_1["DiffSoldeDest"] 
data_1["step_day"] = np.round(data_1["step"]/24).astype(int)

# Suppression des variables inutiles
data_1.drop(columns=["step","isFlaggedFraud","nameOrig","nameDest"], inplace=True) 

# Conversiond des variables en catégories
var_categorielle = ["isFraud", "DiffSoldeOrig", "DiffSoldeDest", "isSameAccount", "activiteAnormale","type"]
for var in var_categorielle:
    data_1[var] = data_1[var].astype("category")

In [ ]:
# Resultats des analyses
if __name__ == "__main__":
    print("\n🔎 DÉBUT DE L'ANALYSE UNIVARIÉE\n")
    # Instanciation de la classe AnalyseUnivariee
    # et lancement de l'analyse univariée
    analyse_1 = AnalyseUnivariee(data_1[["DiffSoldeOrig", "DiffSoldeDest", "isSameAccount", "activiteAnormale"]])
    analyse_2 = BivariateAnalysis(data_1[["DiffSoldeOrig", "DiffSoldeDest", "isSameAccount", "activiteAnormale","amount","oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest","type","isFraud"]])



    # Analyses des variables qualitatives
    analyse_1.analyser_quali()
    analyse_2.BivariateQual()
    analyse_2.BivariateQuantQual()

**4.2. Analyse Multivariée après le Feature Engineering**

*   Encodage
*   Normalisation
*   Définition du target et des features
*   Instanciation de PCA

In [ ]:
# Encoding des variables catégorielles
df_demmies = pd.get_dummies(data_1[["type"]], prefix=["type"], prefix_sep="_", dtype=int)
df = pd.concat([data_1,df_demmies], axis=1)
df.drop(columns=["type"], inplace = True)

# Normalisation des varaibles
df[[
    'amount','oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest','step_day']] = (df[['amount','oldbalanceOrg', 'newbalanceOrig','oldbalanceDest', 'newbalanceDest','step_day']] - df[['amount','oldbalanceOrg', 'newbalanceOrig','oldbalanceDest', 'newbalanceDest',
'step_day']].mean())/df[['amount','oldbalanceOrg', 'newbalanceOrig','oldbalanceDest', 'newbalanceDest','step_day']].std()
df_vf = df.copy()

In [ ]:
an_after = MultivariateAnalyser(df_vf)
an_after.lancer_analyse()

**Instancier le modèle**

In [ ]:
# Définition du train et test
features = df_vf.drop(columns="isFraud")
target = df_vf["isFraud"]
X_train,X_test, y_train, y_test = train_test_split(features,target, train_size=0.3, random_state=42)

In [ ]:
X_test.columns.to_list()

In [ ]:
# Instancier le modèle
model_xgb = XGBClassifier(
    n_estimators = 100, 
    n_jobs = -1,
    learning_rate = 0.1,
    max_depth = 6,
    subsample = 0.8,
    enable_categorical = True)
# Entrainement du modèle
model_xgb.fit(X_train,y_train)

# Prediction du modèle
y_pred = model_xgb.predict(X_test)


In [ ]:
def EvalutionModele(y_true, y_pred):
    # Matrice de confusion
    print(f"\n La matrice de confusion \n")
    matrice_confusion = tabulate(pd.DataFrame(
        confusion_matrix(y_true=y_true, y_pred=y_pred)),
        headers=["Faux Negatif", "Vraie Negatif"],tablefmt = "rounded_grid")
    print(matrice_confusion)


    # Rapport d'évaluation du modèle
    print(f"\n Les indicateurs de performances \n")
    print(classification_report(y_test, y_pred))

**Pipeline du Modèle**

In [ ]:
# Définition du train et test
X = data_1.drop(columns="isFraud")
y = data_1["isFraud"]
X_train,X_test, y_train, y_test = train_test_split(X,y, train_size=0.3, random_state=42)


# Pretraitement des données
preprocessing = ColumnTransformer([
    ("cat",OneHotEncoder(handle_unknown='ignore'),['type']),
    ("norm", StandardScaler(),['amount','oldbalanceOrg', 'newbalanceOrig','oldbalanceDest', 'newbalanceDest','step_day'])
],remainder='passthrough')

# Inplementation du pipeline
pipe_xgboost = Pipeline([
    ('prepros', preprocessing),
    ('model',XGBClassifier(
    n_estimators = 100, 
    n_jobs = -1,
    learning_rate = 0.1,
    max_depth = 6,
    subsample = 0.8,
    enable_categorical = True))
    ])


# Entraitement final
pipe_xgboost.fit(X_train, y_train)

In [ ]:
# Prediction du modèle
y_predite = pipe_xgboost.predict(X_test)

In [ ]:
# Evalution du modèle
EvalutionModele(y_test,y_predite)

**L'importance de chaque variables dans le resultat de la prediction du modèle**

In [ ]:
# Extraire le preprocessor
preprocessor = pipe_xgboost.named_steps["prepros"]

# Transformer X_test pour qu’il soit numérique
X_transformed = preprocessor.transform(X_test)

# Utiliser X_transformed sur le modèle XGBoost
model = pipe_xgboost.named_steps["model"]
explainer = shap.Explainer(model, X_transformed)

shap_values = explainer(X_transformed)
shap.plots.waterfall(shap_values[0])

<!--import pandas as pd

def get_feature_names(column_transformer):
    output_features = []  

    for name, transformer, columns in column_transformer.transformers_:
        if transformer == "drop": 
            continue
        if transformer == "passthrough":
            if isinstance(columns, slice):
                cols = column_transformer._feature_names_in_[columns]
            else:
                cols = columns
            output_features.extend(cols)
        else:
            try:
                if hasattr(transformer, "get_feature_names_out"):
                    names = transformer.get_feature_names_out(columns)
                elif hasattr(transformer, "get_feature_names"):
                    names = transformer.get_feature_names(columns)
                else:
                    names = columns
                output_features.extend(names)
            except:
                output_features.extend(columns)
    return output_features

# Extraire le preprocessor
preprocessor = pipe_xgboost.named_steps["prepros"]

# Récupérer les noms de features

feature_names = get_feature_names(preprocessor)

# Recréer un DataFrame avec noms
X_transformed_df = pd.DataFrame(X_transformed, columns=feature_names)

explainer = shap.Explainer(model, X_transformed_df)
shap_values = explainer(X_transformed_df)
shap.plots.waterfall(shap_values[0])-->